# Experiment 9 — Repeated Entity-disjoint Generalisation Evaluation

**Question:** How do Random, hostname-disjoint and registered-domain-disjoint
evaluation change the estimated performance of two QR phishing detectors?

- Uses the same 9,713 aligned QR–URL pairs as Exp2/Exp6/Exp8.
- Re-trains both the 10-feature lexical XGBoost and Trad-style raw-pixel XGBoost
  **for every split and every random seed**. No old frozen model is reused.
- Each split targets ~20% test rows with similar class prevalence. The split is
  chosen **only by group identity, row count, and labels**; no model scores.
- Strictly checks hostname and registered-domain overlaps; reports intentional
  overlap under random or hostname-only splits.
- Uses an offline ICANN Public Suffix List snapshot when available, and audits
  the precomputed `domain` column. No Internet or API access needed.
- Outputs per-run scores, split audits, sample-level predictions, paired
  registered-domain-bootstrap intervals, figures and reproducibility metadata.

**Important interpretation limits:**
- Samples and class compositions vary across splitting protocols. Differences
  are associations with evaluation conditions, not identified leakage effects.
- This is repeated re-partitioning of a *previously studied* dataset, NOT a new
  untouched external test or a nested estimate of model/feature selection.
- The 10-feature subset was selected earlier; it is **fixed** here. The
  retrospective feature selection was not re-run inside each training fold.
- AUC contrasts in the bootstrap are matched within each split (lexical minus
  raw pixel), not between distinct test partitions.

Required data:
`qr_full_feature_dataset_v2.csv`, `phase14B_final_robust_features.json`,
`qr_codes_29.pickle`, `qr_codes_29_labels.pickle`.
Optional: `esann_icann_psl_snapshot.txt` (included in the downloadable ZIP),
or `ESANN_Corrected_Experiments_Full_Package.zip` containing the same snapshot.
Do not open untrusted pickle files.

## 1. Configuration and imports
Change only the settings below, and run this notebook from the first cell.

In [ ]:
from pathlib import Path
import json, pickle, time, platform, sys, zipfile, warnings
from collections import defaultdict
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.metrics import roc_auc_score, f1_score, matthews_corrcoef, accuracy_score, precision_score, recall_score
from xgboost import XGBClassifier
import sklearn, xgboost

# Recommended 3 seeds; if affordable extend to (42, 43, 44, 45, 46).
P9_SEEDS = (42, 43, 44)
P9_TEST_FRACTION = 0.20
P9_SPLIT_CANDIDATES = 80     # 80 label/size-balanced group-split candidates per seed
P9_BOOTSTRAPS = 300          # 300 is sufficient for a first run; 1000 for final plots
P9_RUN_RAW_PIXEL = True      # True: compare raw XGB and lexical XGB; False: lexical only
P9_N_JOBS = 4
P9_RANDOM_STATE = 42         # seed for deterministic configuration; split varies above
P9_THRESHOLD = 0.50         # fixed threshold for F1/MCC; never tune on test data
P9_SAVE_PREDICTIONS = True

# For diagnostic smoke tests ONLY (not paper results):
# set P9_SEEDS=(42,), P9_BOOTSTRAPS=30, P9_RAW_PARAMS['n_estimators']=20.

# All input files may be set explicitly here if auto-locate fails.
P9_PATH_OVERRIDES = {
    'qr_full_feature_dataset_v2.csv': None,
    'phase14B_final_robust_features.json': None,
    'qr_codes_29.pickle': None,
    'qr_codes_29_labels.pickle': None,
    'esann_icann_psl_snapshot.txt': None,  # optional (CSV domain fallback if missing)
}
P9_ROOTS = [
    Path.cwd(), Path.cwd().parent,
    (Path(__file__).resolve().parent if "__file__" in globals() else Path.cwd()),
    Path.home() / 'Desktop' / 'FYP',
    Path.home() / 'Desktop' / 'FYP' / 'QuishingDataset',
    Path.home() / 'OneDrive' / 'Desktop' / 'FYP',
    Path.home() / 'OneDrive' / 'Desktop' / 'FYP' / 'QuishingDataset',
    Path.home() / 'Documents',
    Path(r'C:\Users\LENOVO\Desktop\FYP'),
    Path(r'C:\Users\LENOVO\Desktop\FYP\QuishingDataset'),
    Path(r'C:\Users\LENOVO\FYP\QuishingDataset'),
]
P9_OUT = Path.cwd() / 'experiment9_entity_disjoint_results'
P9_OUT.mkdir(parents=True, exist_ok=True)

# Match the original Exp6 training configurations, not a fresh hyperparameter
# search chosen on the new test sets. The lexical hyperparameters reproduce the
# earlier frozen model's published configuration but are RE-FITTED per split.
P9_LEX_PARAMS = dict(
    n_estimators=400, learning_rate=0.05, max_depth=4,
    subsample=0.9, colsample_bytree=1.0,
    objective='binary:logistic', eval_metric='logloss',
    tree_method='hist', n_jobs=P9_N_JOBS,
)
P9_RAW_PARAMS = dict(
    n_estimators=150, learning_rate=0.2,
    objective='binary:logistic', eval_metric='logloss',
    tree_method='hist', n_jobs=P9_N_JOBS,
)
assert len(P9_SEEDS) >= 1 and len(P9_SEEDS) == len(set(P9_SEEDS))
assert 0 < P9_TEST_FRACTION < 0.5
assert P9_SPLIT_CANDIDATES >= 1 and P9_BOOTSTRAPS >= 1

## 2. Locate source files and verify aligned QR–URL data
`original_index` is the ONLY allowed mapping from cleaned rows to QR images.

In [ ]:
def p9_locate(name, optional=False):
    val = P9_PATH_OVERRIDES.get(name)
    if val:
        f = Path(val).expanduser()
        if not f.is_file():
            raise FileNotFoundError(f'Invalid override for {name}: {f}')
        return f.resolve()
    for root in P9_ROOTS:
        for f in (root / name, root / 'QuishingDataset' / name,
                  root / 'experiment6_matched_trad_vs_lexical' / name):
            if f.is_file():
                return f.resolve()
    if optional:
        return None
    raise FileNotFoundError(f'File {name!r} not found. Edit P9_PATH_OVERRIDES. '\
                            f'Current directory: {Path.cwd()}')

P9_FILES = {k:p9_locate(k, optional=k=='esann_icann_psl_snapshot.txt')
            for k in P9_PATH_OVERRIDES}
for k,v in P9_FILES.items():
    print(f'{k}: {v or "(optional: will try ZIP or use precomputed domain column)"}')

p9_df = pd.read_csv(P9_FILES['qr_full_feature_dataset_v2.csv'])
with open(P9_FILES['phase14B_final_robust_features.json'],encoding='utf-8') as f:
    p9_features = list(json.load(f)['features'])
assert len(p9_df) == 9713, f'Unexpected cleaned dataset size: {len(p9_df)}'
assert len(p9_features) == 10 and len(set(p9_features)) == 10
needed = ['original_index','url','hostname','domain','label'] + p9_features
assert set(needed).issubset(p9_df.columns), set(needed)-set(p9_df.columns)
assert p9_df[needed].notna().all().all(), 'Missing required values: fix before training'
assert p9_df['original_index'].is_unique
assert p9_df['label'].isin([0,1]).all()
assert np.isfinite(p9_df[p9_features].to_numpy(dtype=float)).all()

p9_y = p9_df['label'].to_numpy(dtype=np.int8)
p9_orig = p9_df['original_index'].to_numpy(dtype=int)
p9_hostname = p9_df['hostname'].astype(str).str.strip().str.lower().str.rstrip('.').to_numpy()
assert (pd.Series(p9_hostname).str.len() > 0).all()
# Input feature order must match the previously selected 10-feature specification.
p9_X_lex = p9_df[p9_features].copy()

with open(P9_FILES['qr_codes_29_labels.pickle'],'rb') as f:
    p9_qr_labels = np.asarray(pickle.load(f),dtype=int)
assert p9_qr_labels.shape == (9987,)
assert (p9_orig >= 0).all() and (p9_orig < len(p9_qr_labels)).all()
assert np.array_equal(p9_qr_labels[p9_orig],p9_y), 'QR/URL label alignment FAILED'

if P9_RUN_RAW_PIXEL:
    with open(P9_FILES['qr_codes_29.pickle'],'rb') as f:
        p9_qr = np.asarray(pickle.load(f))
    assert p9_qr.shape == (9987,69,69),p9_qr.shape
    assert np.isin(p9_qr,[0,1]).all(), 'Expected binary original modules'
    # Model sees original native raw QR modules, not regenerated QR codes.
    p9_X_raw = p9_qr[p9_orig].reshape(len(p9_df),4761).astype(np.float32)
    del p9_qr
else:
    p9_X_raw = None
print(f'PASS: n={len(p9_df)}, labels={np.bincount(p9_y)}, '
      f'unique hostnames={len(np.unique(p9_hostname))}, features={p9_features}')

## 3. Determine registered domains with an offline PSL audit
Uses the same ICANN PSL-snapshot rule as the earlier deployment audit,
including the explicit `fed.us` compatibility rule. If the snapshot is absent,
the original cleaned CSV `domain` column is used, with a visible warning.
The supplied CSV domain groups are NOT silently replaced without reporting.

In [ ]:
def p9_snapshot_text():
    if P9_FILES['esann_icann_psl_snapshot.txt']:
        return P9_FILES['esann_icann_psl_snapshot.txt'].read_text(encoding='utf8'), 'individual PSL snapshot'
    for root in P9_ROOTS:
        zpath=root / 'ESANN_Corrected_Experiments_Full_Package.zip'
        if zpath.exists():
            with zipfile.ZipFile(zpath) as z:
                if 'esann_icann_psl_snapshot.txt' in z.namelist():
                    return z.read('esann_icann_psl_snapshot.txt').decode('utf8'),str(zpath)
    return None, None

p9_psl_text,p9_psl_source=p9_snapshot_text()
if p9_psl_text:
    p9_lines=[s.strip().lower() for s in p9_psl_text.splitlines()
              if s.strip() and not s.strip().startswith('//')]
    p9_exact={s for s in p9_lines if not s.startswith(('*.','!'))}
    p9_wild={s[2:] for s in p9_lines if s.startswith('*.')}
    p9_except={s[1:] for s in p9_lines if s.startswith('!')}
    def p9_registered_domain(host):
        host=str(host).lower().strip('.')
        if host == 'fed.us' or host.endswith('.fed.us'):
            return 'fed.us'
        labels=host.split('.')
        suffix_len=1
        for i in range(len(labels)):
            part='.'.join(labels[i:])
            if part in p9_except:
                suffix_len=len(labels)-i-1
                break
            if part in p9_exact:
                suffix_len=max(suffix_len,len(labels)-i)
            if i+1 < len(labels) and '.'.join(labels[i+1:]) in p9_wild:
                suffix_len=max(suffix_len,len(labels)-i)
        return '.'.join(labels[-suffix_len-1:]) if len(labels)>suffix_len else host
    p9_domain=np.asarray([p9_registered_domain(x) for x in p9_hostname])
    domain_source='offline_ICANN_PSL_snapshot_with_fed_us_compatibility'
    existing=p9_df.domain.astype(str).str.lower().str.strip().to_numpy()
    mismatches=p9_domain!=existing
    p9_domain_audit=p9_df.loc[mismatches,['original_index','hostname','domain']].copy()
    p9_domain_audit['psl_domain']=p9_domain[mismatches]
    p9_domain_audit.to_csv(P9_OUT/'domain_parser_mismatch_audit.csv',index=False)
    print('PSL source:',p9_psl_source,'| CSV domain differences:',int(mismatches.sum()))
    assert mismatches.mean()<.01, 'Too many domain differences; inspect audit before training'
else:
    warnings.warn('PSL snapshot unavailable: using PRECOMPUTED domain column. '
                  'For publication-grade registered-domain analysis, place the '
                  'bundled esann_icann_psl_snapshot.txt alongside this notebook.')
    p9_domain=p9_df.domain.astype(str).str.lower().str.strip().to_numpy()
    domain_source='precomputed_CSV_domain_column_NO_independent_PSL_check'
    p9_domain_audit=pd.DataFrame(columns=['original_index','hostname','domain','psl_domain'])
    p9_domain_audit.to_csv(P9_OUT/'domain_parser_mismatch_audit.csv',index=False)
assert all(bool(d) for d in p9_domain)
assert pd.DataFrame({'hostname':p9_hostname,'domain':p9_domain}).groupby('hostname').domain.nunique().max()==1
print('Registered-domain groups:',len(np.unique(p9_domain)), '| source:',domain_source)

## 4. Construct 3 × N_SEEDS comparable splits, with strict overlap audits
Candidate group splits are chosen by closeness to the requested test fraction
and label prevalence. NO model predictions are used to choose splits.
A random split intentionally allows hostname/registered-domain overlap.

In [ ]:
def p9_make_split(regime, seed):
    idx=np.arange(len(p9_y))
    if regime=='random_stratified':
        tr,te=train_test_split(idx, test_size=P9_TEST_FRACTION,
                               stratify=p9_y,random_state=int(seed))
        return np.sort(tr),np.sort(te),1
    if regime=='hostname_disjoint':
        groups=p9_hostname
    elif regime=='registered_domain_disjoint':
        groups=p9_domain
    else:
        raise ValueError(regime)
    splitter=GroupShuffleSplit(n_splits=P9_SPLIT_CANDIDATES,
                               test_size=P9_TEST_FRACTION,random_state=int(seed))
    best=None
    for candidate, (tr,te) in enumerate(splitter.split(idx,p9_y,groups=groups),1):
        if len(np.unique(p9_y[tr])) != 2 or len(np.unique(p9_y[te])) != 2:
            continue
        frac=abs(len(te)/len(idx)-P9_TEST_FRACTION)
        prevalence=abs(float(p9_y[te].mean())-float(p9_y.mean()))
        score=frac + prevalence # both differences expressed as fractions
        if best is None or score<best[0]:
            best=(score,np.sort(tr),np.sort(te),candidate)
    if best is None:
        raise ValueError(f'No valid two-class {regime} test split for seed={seed}')
    return best[1],best[2],best[3]

P9_REGIMES=['random_stratified','hostname_disjoint','registered_domain_disjoint']
p9_split_map={}
p9_audit_rows=[]
p9_split_rows=[]
for regime in P9_REGIMES:
    for seed in P9_SEEDS:
        tr,te,candidate=p9_make_split(regime,seed)
        assert len(set(tr)&set(te))==0 and len(tr)+len(te)==len(p9_y)
        assert len(np.unique(p9_y[tr]))==2 and len(np.unique(p9_y[te]))==2
        overlap_host=len(set(p9_hostname[tr]) & set(p9_hostname[te]))
        overlap_domain=len(set(p9_domain[tr]) & set(p9_domain[te]))
        if regime=='hostname_disjoint':
            assert overlap_host==0, 'Hostname leakage!'
        if regime=='registered_domain_disjoint':
            assert overlap_host==0 and overlap_domain==0, 'Domain leakage!'
        p9_split_map[(regime,int(seed))]=(tr,te)
        p9_audit_rows.append(dict(regime=regime,seed=seed,
             selected_candidate_number=candidate,train_n=len(tr),test_n=len(te),
             test_fraction=len(te)/len(p9_y),
             train_phishing_rate=float(p9_y[tr].mean()),test_phishing_rate=float(p9_y[te].mean()),
             train_hostname_n=len(set(p9_hostname[tr])),test_hostname_n=len(set(p9_hostname[te])),
             train_domain_n=len(set(p9_domain[tr])),test_domain_n=len(set(p9_domain[te])),
             hostname_overlap=overlap_host,registered_domain_overlap=overlap_domain))
        for role,part in [('train',tr),('test',te)]:
            p9_split_rows.extend({'regime':regime,'seed':seed,'role':role,
                                  'original_index':int(p9_orig[j]),'label':int(p9_y[j]),
                                  'hostname':p9_hostname[j],'registered_domain':p9_domain[j]}
                                  for j in part)
p9_split_audit=pd.DataFrame(p9_audit_rows)
p9_split_audit.to_csv(P9_OUT/'split_integrity_audit.csv',index=False)
pd.DataFrame(p9_split_rows).to_csv(P9_OUT/'split_membership.csv',index=False)
print('\n=== SPLIT AUDIT (strict group conditions verified) ===')
print(p9_split_audit[['regime','seed','train_n','test_n','test_phishing_rate',
                      'hostname_overlap','registered_domain_overlap']].round(4).to_string(index=False))

## 5. Retrain lexical XGB and raw-pixel XGB from scratch for each split
Each model is fitted ONLY on that split's train rows. Fixed hyperparameters,
fixed 0.5 decision threshold, no test-based feature selection or tuning.

In [ ]:
def p9_scores(y,scores):
    scores=np.asarray(scores,dtype=float)
    y=np.asarray(y,dtype=int)
    assert len(scores)==len(y) and np.isfinite(scores).all()
    pred=(scores>=P9_THRESHOLD).astype(int)
    return dict(auc=float(roc_auc_score(y,scores)),
                f1=float(f1_score(y,pred,zero_division=0)),
                mcc=float(matthews_corrcoef(y,pred)),
                accuracy=float(accuracy_score(y,pred)),
                precision=float(precision_score(y,pred,zero_division=0)),
                recall=float(recall_score(y,pred,zero_division=0)),
                predicted_phishing_rate=float(pred.mean()),
                score_std=float(np.std(scores)))

p9_metric_rows=[]
p9_prediction_rows=[]
p9_pred_cache={}
p9_run_start=time.perf_counter()
for regime in P9_REGIMES:
    for seed in P9_SEEDS:
        tr,te=p9_split_map[(regime,int(seed))]
        ytr,yte=p9_y[tr],p9_y[te]
        model_scores={}
        for name,X,params in [('lexical_10',p9_X_lex,P9_LEX_PARAMS),
                              ('raw_pixel',p9_X_raw,P9_RAW_PARAMS)]:
            if name=='raw_pixel' and not P9_RUN_RAW_PIXEL:
                continue
            assert X is not None
            model=XGBClassifier(**params,random_state=int(seed))
            t0=time.perf_counter()
            model.fit(X.iloc[tr] if isinstance(X,pd.DataFrame) else X[tr],ytr)
            secs=time.perf_counter()-t0
            scores=model.predict_proba(X.iloc[te] if isinstance(X,pd.DataFrame) else X[te])[:,1]
            model_scores[name]=scores
            metr=p9_scores(yte,scores)
            p9_metric_rows.append(dict(regime=regime,seed=seed,model=name,
                                       train_n=len(tr),test_n=len(te),
                                       test_phishing_rate=float(yte.mean()),
                                       train_seconds=secs,**metr))
            print(f'{regime:28s} | seed {seed} | {name:10s} | '
                  f'AUC={metr["auc"]:.4f} F1={metr["f1"]:.4f} '
                  f'MCC={metr["mcc"]:.4f} | fit {secs:.1f}s',flush=True)
        p9_pred_cache[(regime,int(seed))]=dict(
            y=yte,domain=p9_domain[te],hostname=p9_hostname[te],
            scores=model_scores,test_index=te)
        if P9_SAVE_PREDICTIONS:
            for name,scores in model_scores.items():
                p9_prediction_rows.extend(
                    dict(regime=regime,seed=seed,model=name,
                         original_index=int(p9_orig[j]),label=int(p9_y[j]),
                         hostname=p9_hostname[j],registered_domain=p9_domain[j],
                         prediction_probability=float(score),
                         prediction=int(score>=P9_THRESHOLD))
                    for j,score in zip(te,scores))
print(f'\nCompleted all training fits in {time.perf_counter()-p9_run_start:.1f}s')
p9_runs=pd.DataFrame(p9_metric_rows)
p9_runs.to_csv(P9_OUT/'metrics_by_split_seed_model.csv',index=False)
if P9_SAVE_PREDICTIONS:
    pd.DataFrame(p9_prediction_rows).to_csv(P9_OUT/'sample_level_predictions.csv',index=False)
print('\n=== TEST SCORES (AUC / F1 / MCC) ===')
print(p9_runs[['regime','seed','model','test_n','auc','f1','mcc']].round(4).to_string(index=False))

## 6. Aggregate across random seeds (mean ± sample standard deviation)
These are DESCRIPTIVE stability summaries from repeated partitions, not an
independent-samples confidence interval or proof of causal leakage.

In [ ]:
p9_agg=p9_runs.groupby(['regime','model'],as_index=False).agg(
    runs=('auc','size'),
    auc_mean=('auc','mean'),auc_sd=('auc','std'),
    f1_mean=('f1','mean'),f1_sd=('f1','std'),
    mcc_mean=('mcc','mean'),mcc_sd=('mcc','std'),
    accuracy_mean=('accuracy','mean'),accuracy_sd=('accuracy','std'),
    test_n_mean=('test_n','mean'),
    test_phishing_rate_mean=('test_phishing_rate','mean'),
)
p9_agg.to_csv(P9_OUT/'summary_mean_sd_by_regime_model.csv',index=False)
print('\n=== AGGREGATED (DESCRIPTIVE, NOT INDEPENDENT 95% CI) ===')
print(p9_agg.round(4).to_string(index=False))

## 7. Paired bootstrap on each split: lexical minus raw-pixel AUC
The paired models see exactly the SAME test instances in the run. Resample
registered domains as clusters to respect related URL observations. Do not
interpret these as CIs for the differences BETWEEN evaluation protocols.

In [ ]:
p9_boot_rows=[]
if P9_RUN_RAW_PIXEL:
    for regime in P9_REGIMES:
        for seed in P9_SEEDS:
            pack=p9_pred_cache[(regime,int(seed))]
            y=pack['y']; d=pack['domain']; sc=pack['scores']
            if not ('lexical_10' in sc and 'raw_pixel' in sc):
                continue
            delta_obs=float(roc_auc_score(y,sc['lexical_10'])-roc_auc_score(y,sc['raw_pixel']))
            unique,inv=np.unique(d,return_inverse=True)
            blocks=[np.flatnonzero(inv==i) for i in range(len(unique))]
            rng=np.random.default_rng(73111+int(seed)+1000*P9_REGIMES.index(regime))
            draws=[]
            for _ in range(P9_BOOTSTRAPS):
                selected=rng.integers(0,len(unique),size=len(unique))
                sub=np.concatenate([blocks[k] for k in selected])
                if len(np.unique(y[sub]))<2:
                    continue
                diff=roc_auc_score(y[sub],sc['lexical_10'][sub])-roc_auc_score(y[sub],sc['raw_pixel'][sub])
                draws.append(float(diff))
            assert len(draws)>=max(1,int(.90*P9_BOOTSTRAPS)), 'Too few valid bootstrap resamples'
            lo,hi=np.quantile(draws,[.025,.975])
            p9_boot_rows.append(dict(regime=regime,seed=seed,
                 auc_delta_lexical_minus_raw=delta_obs,ci95_lower=float(lo),ci95_upper=float(hi),
                 cluster='registered_domain',valid_bootstraps=len(draws)))
p9_paired=pd.DataFrame(p9_boot_rows)
p9_paired.to_csv(P9_OUT/'paired_auc_cluster_bootstrap_ci.csv',index=False)
if len(p9_paired):
    print('\n=== PAIRED REGISTERED-DOMAIN CLUSTER BOOTSTRAP AUC DELTAS ===')
    print(p9_paired.round(4).to_string(index=False))

## 8. Figures for manuscript screening (and GitHub evidence)
One plot for mean AUC ± seed SD, one for the lexical–visual paired AUC gap.

In [ ]:
order=P9_REGIMES
models=['lexical_10']+(['raw_pixel'] if P9_RUN_RAW_PIXEL else [])
fig,ax=plt.subplots(figsize=(9,4.6))
xs=np.arange(len(order)); width=.34 if len(models)==2 else .55
for k,model in enumerate(models):
    sub=p9_agg.set_index(['regime','model'])
    ms=np.array([sub.loc[(r,model),'auc_mean'] for r in order]);
    sd=np.nan_to_num(np.array([sub.loc[(r,model),'auc_sd'] for r in order]))
    loc=xs+(k-(len(models)-1)/2)*width
    bars=ax.bar(loc,ms,width,yerr=sd,capsize=4,label=model)
    for bar,val in zip(bars,ms):
        ax.text(bar.get_x()+bar.get_width()/2,bar.get_height()+.02,f'{val:.3f}',
                ha='center',va='bottom',fontsize=9)
ax.set_xticks(xs, ['Random','Hostname-disjoint','Registered-domain-disjoint'])
ax.set_ylabel('ROC–AUC (mean ± SD across seeds)')
ax.set_ylim(0,1.09);ax.set_title('Experiment 9: Evaluation protocol sensitivity')
ax.legend();ax.grid(axis='y',alpha=.2);fig.tight_layout()
fig.savefig(P9_OUT/'auc_by_split_protocol.png',dpi=170,bbox_inches='tight')
plt.show()

if P9_RUN_RAW_PIXEL:
    fig,ax=plt.subplots(figsize=(8.6,4.2))
    for seed in P9_SEEDS:
        r=p9_paired.set_index(['regime','seed'])
        vals=[r.loc[(reg,seed),'auc_delta_lexical_minus_raw'] for reg in order]
        ax.plot(range(len(order)),vals,marker='o',label=f'Seed {seed}')
    ax.axhline(0,lw=1,color='gray')
    ax.set_xticks(range(len(order)),['Random','Hostname','Registered domain'])
    ax.set_ylabel('AUC difference (lexical − raw-pixel)')
    ax.set_title('Matched AUC difference for each protocol and seed')
    ax.legend();ax.grid(alpha=.2);fig.tight_layout()
    fig.savefig(P9_OUT/'lexical_minus_raw_auc_by_seed.png',dpi=170,bbox_inches='tight')
    plt.show()

## 9. Save protocol, checks, and a concise summary for the paper
Compare these scores with previous experiments carefully: this uses repeated
retraining on NEW partitions and is not the fixed Exp6 matched holdout.

In [ ]:
p9_meta={
    'experiment':'Exp9 multi-seed entity-disjoint generalisation',
    'n_clean_pairs':len(p9_y),
    'positive_label':1,
    'regimes':P9_REGIMES,
    'seeds':list(P9_SEEDS),
    'test_fraction_target':P9_TEST_FRACTION,
    'split_candidates_per_grouped_seed':P9_SPLIT_CANDIDATES,
    'split_selection':'minimize (abs test fraction error + abs test prevalence error), labels and groups only',
    'registered_domain_source':domain_source,
    'psl_snapshot_source':p9_psl_source,
    'feature_list':p9_features,
    'lexical_params':P9_LEX_PARAMS,
    'raw_params':P9_RAW_PARAMS if P9_RUN_RAW_PIXEL else None,
    'raw_pixel_enabled':P9_RUN_RAW_PIXEL,
    'fixed_class_threshold':P9_THRESHOLD,
    'bootstrap_cluster':'registered_domain',
    'bootstrap_replicates_requested':P9_BOOTSTRAPS,
    'warnings':[
        'Feature subset was previously selected, not nested within each new train split.',
        'Repeated partitions are not independent external tests.',
        'This dataset and earlier holdout have been used in prior development/diagnostics.',
        'Different evaluation protocols have different test entities and can vary in difficulty.',
        'Raw and lexical use different representations/hyperparameters; identical test rows within run.',
    ],
    'python':sys.version.split()[0],
    'platform':platform.platform(),
    'numpy':np.__version__,
    'pandas':pd.__version__,
    'sklearn':sklearn.__version__,
    'xgboost':xgboost.__version__,
    'elapsed_seconds':time.perf_counter()-p9_run_start,
}
with open(P9_OUT/'experiment9_protocol.json','w',encoding='utf8') as f:
    json.dump(p9_meta,f,indent=2,ensure_ascii=False)
print('\nDONE. Outputs:',P9_OUT.resolve())
for path in sorted(P9_OUT.iterdir()):
    print(f'  {path.name} ({path.stat().st_size:,} bytes)')
print('\nREPORTING RULE: label the above as repeated entity-aware generalisation evaluation;')
print('do not assert entity overlap alone CAUSES the AUC changes.')